# Задание 5. Тестирование с pytest и pytest-cov

**Выполнил:** WhyNot-png (kohen).

[Исходный ноутбук преподавателя](https://drive.google.com/file/d/1na-9FxCYq65wm8WWcSnq75eCxzs1MND3/view).
Копия сохраняет исходный `grades.py` без изменений. Ниже показаны реальный исходный
сбой, исправленные тесты, промежуточное и итоговое покрытие, новые проверки и ответы.
Все этапы можно повторить командой **Среда выполнения → Перезапустить сеанс и выполнить все**.
Ожидаемые падения исходных тестов перехвачены и не останавливают ноутбук.

## Шаг 0. Установка и отдельный рабочий каталог

In [1]:
%pip install -q pytest==9.1.1 pytest-cov==7.1.0


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

# Repeated execution stays in the same directory, not lab5_grades/lab5_grades.
if Path.cwd().name != "lab5_grades":
    Path("lab5_grades").mkdir(exist_ok=True)
    os.chdir("lab5_grades")
print("Python:", sys.version.split()[0])
print("Рабочий каталог:", Path.cwd().name)


def run_pytest(*arguments, expected=0):
    completed = subprocess.run(
        [sys.executable, "-m", "pytest", *arguments],
        text=True, encoding="utf-8", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    print(completed.stdout)
    print("Код завершения:", completed.returncode)
    assert completed.returncode == expected, completed.stdout
    return completed

Python: 3.12.12
Рабочий каталог: lab5_grades


## Шаг 1. Исходный модуль — не изменён

In [3]:
%%writefile grades.py
"""Небольшой модуль для работы с оценками студентов.

ВАЖНО: код этого файла менять нельзя — ошибки находятся в тестах!
"""


def average(scores):
    """Среднее арифметическое списка оценок."""
    if not scores:
        raise ValueError("Список оценок пуст")
    return sum(scores) / len(scores)


def letter_grade(score):
    """Переводит балл (0-100) в буквенную оценку A-F."""
    if score < 0 or score > 100:
        raise ValueError("Балл должен быть от 0 до 100")
    if score >= 90:
        return "A"
    if score >= 80:
        return "B"
    if score >= 70:
        return "C"
    if score >= 60:
        return "D"
    return "F"


def is_passed(score, threshold=60):
    """Сдан ли предмет: балл не ниже порога."""
    return score >= threshold


def normalize(scores):
    """Приводит баллы к шкале от 0 до 1 (min-max нормализация)."""
    if not scores:
        raise ValueError("Список оценок пуст")
    lo, hi = min(scores), max(scores)
    if lo == hi:
        return [0.0 for _ in scores]
    return [(s - lo) / (hi - lo) for s in scores]


def top_n(students, n):
    """Возвращает n лучших студентов как список пар (имя, балл)."""
    if n <= 0:
        raise ValueError("n должно быть положительным")
    return sorted(students.items(), key=lambda item: item[1], reverse=True)[:n]


def count_by_grade(scores):
    """Считает, сколько раз встречается каждая буквенная оценка."""
    result = {}
    for score in scores:
        grade = letter_grade(score)
        result[grade] = result.get(grade, 0) + 1
    return result


def apply_bonus(score, bonus):
    """Добавляет бонусные баллы, но не больше 100."""
    if bonus < 0:
        raise ValueError("Бонус не может быть отрицательным")
    return min(score + bonus, 100)

Overwriting grades.py


In [4]:
ORIGINAL_GRADES_SHA256 = "c5cf89255a91489baf70da51676b3a22082b9e049423c3af83f2c8b1d1c15357"
assert hashlib.sha256(Path("grades.py").read_bytes()).hexdigest() == ORIGINAL_GRADES_SHA256
print("grades.py совпадает с исходным файлом, SHA-256:", ORIGINAL_GRADES_SHA256)

grades.py совпадает с исходным файлом, SHA-256: c5cf89255a91489baf70da51676b3a22082b9e049423c3af83f2c8b1d1c15357


## Шаг 2. Исходные тесты и два диагностических запуска

Первый запуск останавливается на неверном импорте. Затем исправляется **только** импорт `count_by_grades`, чтобы увидеть оставшиеся ошибки. Это ещё не итоговое решение.

In [5]:
%%writefile test_grades.py
import pytest
from grades import average, letter_grade, is_passed, normalize, top_n, count_by_grades


def test_average():
    assert average([80, 90, 100]) == 80


def test_average_empty():
    with pytest.raises(KeyError):
        average([])


def test_letter_grade_a():
    assert letter_grade(95) == "B"


def test_letter_grade_boundary():
    assert letter_grade(90) == "B"


def test_is_passed():
    assert is_passed(59)


def test_normalize():
    assert normalize([10, 20, 30]) == [0.0, 0.5, 1.0]


def test_normalize_equal_values():
    normalize([5, 5, 5])


def test_top_n():
    students = {"Аня": 70, "Борис": 95, "Вера": 85}
    assert top_n(students, 1) == ["Борис"]


def check_apply_bonus():
    assert apply_bonus(95, 10) == 100

Overwriting test_grades.py


In [6]:
original_tests = Path("test_grades.py").read_text(encoding="utf-8")
Path("test_grades_original.txt").write_text(original_tests, encoding="utf-8")
run_pytest("test_grades.py", "-v", expected=2)

# Reveal assertion failures hidden behind the collection error.
Path("test_grades.py").write_text(
    original_tests.replace("count_by_grades", "count_by_grade"), encoding="utf-8"
)
run_pytest("test_grades.py", "-v", expected=1)

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.7.0 -- /home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python
cachedir: .pytest_cache
rootdir: /vercel/sandbox/data/lab5/lab5_grades
plugins: platformdirs-4.13.5, cov-7.1.0
collecting ... collected 0 items / 1 error

==================================== ERRORS ====================================
_______________________ ERROR collecting test_grades.py ________________________
ImportError while importing test module '/vercel/sandbox/data/lab5/lab5_grades/test_grades.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/home/vercel-sandbox/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
test_grades.py:2: in <module>
    fro

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.7.0 -- /home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python
cachedir: .pytest_cache
rootdir: /vercel/sandbox/data/lab5/lab5_grades
plugins: platformdirs-4.13.5, cov-7.1.0
collecting ... collected 8 items

test_grades.py::test_average FAILED                                      [ 12%]
test_grades.py::test_average_empty FAILED                                [ 25%]
test_grades.py::test_letter_grade_a FAILED                               [ 37%]
test_grades.py::test_letter_grade_boundary FAILED                        [ 50%]
test_grades.py::test_is_passed FAILED                                    [ 62%]
test_grades.py::test_normalize PASSED                                    [ 75%]
test_grades.py::test_normalize_equal_values PASSED                       [ 87%]
test_grades.py::test_top_n FAILED                                        [100%]

======

CompletedProcess(args=['/home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python', '-m', 'pytest', 'test_grades.py', '-v'], returncode=1, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.7.0 -- /home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python\ncachedir: .pytest_cache\nrootdir: /vercel/sandbox/data/lab5/lab5_grades\nplugins: platformdirs-4.13.5, cov-7.1.0\n\x1bcollecting ... \x1bcollected 8 items\n\ntest_grades.py::test_average \x1bFAILED\x1b\x1b                                      [ 12%]\x1b\ntest_grades.py::test_average_empty \x1bFAILED\x1b\x1b                                [ 25%]\x1b\ntest_grades.py::test_letter_grade_a \x1bFAILED\x1b\x1b                               [ 37%]\x1b\ntest_grades.py::test_letter_grade_boundary \x1bFAILED\x1b\x1b                        [ 50%]\x1b\ntest_grades.py::test_is_passed \x1bFAILED\x1b\x1b                                   

## Шаг 3. Исправленные исходные тесты

In [7]:
%%writefile test_grades.py
import pytest
from grades import average, letter_grade, is_passed, normalize, top_n, count_by_grade, apply_bonus


def test_average():
    assert average([80, 90, 100]) == 90


def test_average_empty():
    with pytest.raises(ValueError, match="Список оценок пуст"):
        average([])


def test_letter_grade_a():
    assert letter_grade(95) == "A"


def test_letter_grade_boundary():
    assert letter_grade(90) == "A"


def test_is_passed():
    assert is_passed(59) is False


def test_normalize():
    assert normalize([10, 20, 30]) == [0.0, 0.5, 1.0]


def test_normalize_equal_values():
    assert normalize([5, 5, 5]) == [0.0, 0.0, 0.0]


def test_top_n():
    students = {"Аня": 70, "Борис": 95, "Вера": 85}
    assert top_n(students, 1) == [("Борис", 95)]


def test_apply_bonus():
    assert apply_bonus(95, 10) == 100

Overwriting test_grades.py


In [8]:
run_pytest("test_grades.py", "-v")

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.7.0 -- /home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python
cachedir: .pytest_cache
rootdir: /vercel/sandbox/data/lab5/lab5_grades
plugins: platformdirs-4.13.5, cov-7.1.0
collecting ... collected 9 items

test_grades.py::test_average PASSED                                      [ 11%]
test_grades.py::test_average_empty PASSED                                [ 22%]
test_grades.py::test_letter_grade_a PASSED                               [ 33%]
test_grades.py::test_letter_grade_boundary PASSED                        [ 44%]
test_grades.py::test_is_passed PASSED                                    [ 55%]
test_grades.py::test_normalize PASSED                                    [ 66%]
test_grades.py::test_normalize_equal_values PASSED                       [ 77%]
test_grades.py::test_top_n PASSED                                        [ 88%]
test_gr

CompletedProcess(args=['/home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python', '-m', 'pytest', 'test_grades.py', '-v'], returncode=0, stdout='\x1b============================= test session starts ==============================\x1b\nplatform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.7.0 -- /home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python\ncachedir: .pytest_cache\nrootdir: /vercel/sandbox/data/lab5/lab5_grades\nplugins: platformdirs-4.13.5, cov-7.1.0\n\x1bcollecting ... \x1bcollected 9 items\n\ntest_grades.py::test_average \x1bPASSED\x1b\x1b                                      [ 11%]\x1b\ntest_grades.py::test_average_empty \x1bPASSED\x1b\x1b                                [ 22%]\x1b\ntest_grades.py::test_letter_grade_a \x1bPASSED\x1b\x1b                               [ 33%]\x1b\ntest_grades.py::test_letter_grade_boundary \x1bPASSED\x1b\x1b                        [ 44%]\x1b\ntest_grades.py::test_is_passed \x1bPASSED\x1b\x1b                                   

## Таблица найденных ошибок

| № | Тест / место | В чём ошибка | Исправление |
| --- | --- | --- | --- |
| 1 | Импорт `count_by_grades` | Такого имени нет в модуле; сбор тестов прерывается | Импортировать `count_by_grade` |
| 2 | `test_average` | Для 80, 90, 100 ожидается 80 вместо среднего 90 | Ожидаемое значение заменено на 90 |
| 3 | `test_average_empty` | Ожидается `KeyError`, но контракт требует `ValueError` | `pytest.raises(ValueError)`, проверено сообщение |
| 4 | `test_letter_grade_a` | Для 95 ожидается B | Исправлено на A |
| 5 | `test_letter_grade_boundary` | Для 90 ожидается B, хотя порог A включительный | Исправлено на A |
| 6 | `test_is_passed` | 59 проверялось как проходной балл при пороге 60 | Проверяется `is False` |
| 7 | `test_normalize_equal_values` | Только вызов, результат не проверяется | Добавлен `assert` для трёх нулей |
| 8 | `test_top_n` | Ожидается список имён, хотя возвращаются пары `(имя, балл)` | Ожидается `[("Борис", 95)]` |
| 9 | `check_apply_bonus` | Имя не начинается с `test_`, pytest не собирает функцию | Переименовано в `test_apply_bonus` |
| 10 | `test_apply_bonus` после переименования | `apply_bonus` не импортирован; раньше это скрывалось отсутствием сбора | Добавлен импорт функции |

`test_normalize` в исходнике был правильным и сохранён по смыслу без изменений.
Неиспользованный импорт `count_by_grade` не мешает pytest; функция проверяется новыми тестами ниже.

## Шаг 4. Покрытие после исправления исходных тестов

In [9]:
run_pytest("test_grades.py", "--cov=grades", "--cov-report=term-missing",
           "--cov-report=json:coverage_initial.json")
initial = json.loads(Path("coverage_initial.json").read_text(encoding="utf-8"))
initial_file = initial["files"]["grades.py"]
print("Покрытие после исправления: %.2f%%" % initial["totals"]["percent_covered"])
print("Непокрытые строки:", initial_file["missing_lines"])

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.7.0
rootdir: /vercel/sandbox/data/lab5/lab5_grades
plugins: platformdirs-4.13.5, cov-7.1.0
collected 9 items

test_grades.py .........                                                 [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.12.12-final-0 _______________

Name        Stmts   Miss  Cover   Missing
-----------------------------------------
grades.py      39     16    59%   17, 20-26, 37, 47, 53-57, 63
-----------------------------------------
TOTAL          39     16    59%
Coverage JSON written to file coverage_initial.json
============================== 9 passed in 0.04s ===============================

Код завершения: 0
Покрытие после исправления: 58.97%
Непокрытые строки: [17, 20, 21, 22, 23, 24, 25, 26, 37, 47, 53, 54, 55, 56, 57, 63]


## Результаты измерения покрытия

| Этап | Успешных тестов | Покрыто строк | Покрытие строк | Покрыто ветвей |
| --- | ---: | ---: | ---: | --- |
| Исправлены исходные тесты, дополнительные ещё не включены | 9 | 23 / 39 | 58.97% | Отдельно не измерялось |
| Добавлены собственные тесты | 41 | 39 / 39 | 100% | 22 / 22 (100%) |

**Непокрытые строки после исправления:** `[17, 20, 21, 22, 23, 24, 25, 26, 37, 47, 53, 54, 55, 56, 57, 63]`.

| Функция / ветка | Непокрытые строки до новых тестов | Чем закрыта |
| --- | --- | --- |
| `letter_grade`: ошибка входа и оценки B–F | 17, 20–26 | Параметризация всех интервалов и границ, баллы −1 и 101 |
| `normalize`: пустой список | 37 | `pytest.raises(ValueError)` |
| `top_n`: неположительное n | 47 | Параметры 0 и −1 |
| `count_by_grade`: весь подсчёт | 53–57 | Повторяющиеся оценки, все уровни, пустой список, ошибка оценки |
| `apply_bonus`: отрицательный бонус | 63 | `pytest.raises(ValueError)` |

Итог: непокрытых строк и ветвей нет. Процент получен реальным запуском pytest-cov,
а не вручную подставленным результатом. Покрытие 100% не является доказательством отсутствия ошибок.
Модуль записан исходной `%%writefile`-ячейкой без редактирования;
добавляемый самой магией завершающий перевод строки учтён в контрольной сумме.
SHA-256 файла: `c5cf89255a91489baf70da51676b3a22082b9e049423c3af83f2c8b1d1c15357`.


## Шаг 5. Дополнительные осмысленные тесты

Проверяются все буквенные интервалы и их границы, ошибки входа, пороги сдачи, пустые наборы, нормализация, сортировка и ограничение бонуса.

In [10]:
%%writefile test_extra.py
import pytest
from grades import average, letter_grade, is_passed, normalize, top_n, count_by_grade, apply_bonus


@pytest.mark.parametrize("score", [-1, 101])
def test_letter_grade_invalid_score(score):
    with pytest.raises(ValueError, match="от 0 до 100"):
        letter_grade(score)


@pytest.mark.parametrize("score, expected", [
    (0, "F"), (59, "F"), (60, "D"), (69, "D"), (70, "C"),
    (79, "C"), (80, "B"), (89, "B"), (90, "A"), (100, "A"),
])
def test_letter_grade_all_levels(score, expected):
    assert letter_grade(score) == expected


def test_count_by_grade():
    assert count_by_grade([95, 85, 92]) == {"A": 2, "B": 1}
    assert count_by_grade([95, 85, 75, 65, 55]) == {"A": 1, "B": 1, "C": 1, "D": 1, "F": 1}
    assert count_by_grade([]) == {}


def test_count_by_grade_rejects_invalid_score():
    with pytest.raises(ValueError):
        count_by_grade([95, -1])


@pytest.mark.parametrize("score, bonus, expected", [
    (95, 10, 100), (50, 10, 60), (50, 0, 50), (100, 10, 100),
])
def test_apply_bonus(score, bonus, expected):
    assert apply_bonus(score, bonus) == expected


def test_apply_bonus_negative():
    with pytest.raises(ValueError, match="Бонус не может быть отрицательным"):
        apply_bonus(50, -1)


@pytest.mark.parametrize("score, threshold, expected", [
    (59, 60, False), (60, 60, True), (100, 60, True),
    (69, 70, False), (70, 70, True),
])
def test_is_passed_threshold(score, threshold, expected):
    assert is_passed(score, threshold) is expected


def test_average_fractional_and_singleton():
    assert average([1, 2]) == pytest.approx(1.5)
    assert average([7]) == 7


def test_normalize_empty():
    with pytest.raises(ValueError, match="Список оценок пуст"):
        normalize([])


@pytest.mark.parametrize("scores, expected", [
    ([7], [0.0]), ([-10, 0, 10], [0.0, 0.5, 1.0]),
    ([30, 10, 20], [1.0, 0.0, 0.5]),
])
def test_normalize_values(scores, expected):
    assert normalize(scores) == pytest.approx(expected)


@pytest.mark.parametrize("n", [0, -1])
def test_top_n_invalid_size(n):
    with pytest.raises(ValueError, match="n должно быть положительным"):
        top_n({"Аня": 70}, n)


def test_top_n_order_empty_and_large_n():
    students = {"Аня": 70, "Борис": 95, "Вера": 85}
    snapshot = students.copy()
    assert top_n(students, 2) == [("Борис", 95), ("Вера", 85)]
    assert top_n(students, 10) == [("Борис", 95), ("Вера", 85), ("Аня", 70)]
    assert top_n({}, 1) == []
    assert students == snapshot

Overwriting test_extra.py


## Ответы на четыре вопроса

1. **Самая коварная ошибка.** `check_apply_bonus` вообще не собирался pytest, поэтому
его отсутствие в запуске легко принять за успех. После переименования проявляется
вторая скрытая ошибка — отсутствие импорта `apply_bonus`; отдельно опасен зелёный
`test_normalize_equal_values`, в котором не было проверки результата.

2. **Что означает 100% покрытия?** В используемом отчёте это означает выполнение
всех учитываемых строк модуля хотя бы одним тестом. Оно не гарантирует правильность
ожиданий, проверку всех входных данных или отсутствие ошибок; покрытие ветвей —
отдельная метрика, поэтому финальный запуск также включает `--cov-branch`.

3. **Почему опасен тест без assert?** Простой вызов завершится успешно даже при
неверном результате, если функция не выбросила исключение. Не каждый тест обязан
содержать буквальный `assert`: например, `pytest.raises` тоже проверяет поведение,
но в исходном тесте нормализации не было ни одной такой проверки.

4. **Зачем негативные сценарии?** Они проверяют контракт ошибок: тип исключения,
сообщение и реакцию на недопустимые значения. Иначе функция может правильно работать
на обычных данных, но молча принимать пустой список, неверный балл или отрицательный бонус.

## Шаг 6. Итоговая проверка

Порог 70% enforced через pytest-cov. Дополнительно включено покрытие ветвей и проверяется неизменность исходного модуля. При любой ошибке ячейка завершается исключением, а не только печатает красный лог.

In [11]:
assert hashlib.sha256(Path("grades.py").read_bytes()).hexdigest() == ORIGINAL_GRADES_SHA256
run_pytest("test_grades.py", "test_extra.py", "-v", "--cov=grades", "--cov-branch",
           "--cov-report=term-missing", "--cov-report=json:coverage_final.json",
           "--cov-fail-under=70")
final = json.loads(Path("coverage_final.json").read_text(encoding="utf-8"))
assert final["totals"]["percent_covered"] >= 70
assert not final["files"]["grades.py"]["missing_lines"]
print("Итоговое покрытие строк и ветвей: %.2f%%" % final["totals"]["percent_covered"])
print("grades.py не изменён; финальная проверка выполнена успешно.")

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-9.1.1, pluggy-1.7.0 -- /home/vercel-sandbox/.cache/uv/builds-v0/.tmpVjoFpE/bin/python
cachedir: .pytest_cache
rootdir: /vercel/sandbox/data/lab5/lab5_grades
plugins: platformdirs-4.13.5, cov-7.1.0
collecting ... collected 41 items

test_grades.py::test_average PASSED                                      [  2%]
test_grades.py::test_average_empty PASSED                                [  4%]
test_grades.py::test_letter_grade_a PASSED                               [  7%]
test_grades.py::test_letter_grade_boundary PASSED                        [  9%]
test_grades.py::test_is_passed PASSED                                    [ 12%]
test_grades.py::test_normalize PASSED                                    [ 14%]
test_grades.py::test_normalize_equal_values PASSED                       [ 17%]
test_grades.py::test_top_n PASSED                                        [ 19%]
test_g